In [ ]:
import re
import pandas as pd
import json

### Utils

In [ ]:
# Split and clean multi-item columns
def split_to_array(column_value, delimiter=","):
    if pd.notna(column_value):
        return [item.strip() for item in column_value.split(delimiter)]
    return []

# Clean breeds column
def parse_breeds(breeds_text, species_list):
    if pd.notna(breeds_text):
        matches = re.findall(r"\*\*(.*?)\*\*\s*(.*?)(?=(\*\*|$))", breeds_text)
        species_breeds = {}
        for match in matches:
            species_name = match[0].strip()
            breed = match[1].strip()
            if species_name in species_list:
                species_breeds[species_name] = species_breeds.get(species_name, []) + [breed]
        return species_breeds
    return {}

# Clean markdown-style formatting in locations column
def clean_location(location_text):
    if pd.notna(location_text):
        # Extract the country names and associated locations
        country_location_dict = {}
        # Find all bold country names (**country**) and the following location data
        pattern = r"\*\*(.*?)\*\*\s*([^*]+)"
        matches = re.findall(pattern, location_text)
        for country, location in matches:
            country = country.strip()
            location = location.strip()
            if country not in country_location_dict:
                country_location_dict[country] = []
            country_location_dict[country].append(location)
        return country_location_dict
    return {}

# Clean credits column
def parse_credits(credit_text):
    if pd.notna(credit_text):
        entries = re.split(r"\s*,\s+(?=\*\*)", credit_text.strip())
        cleaned_entries = []
        for entry in entries:
            entry = re.sub(r"\*\*(.*?)\*\*", r"\1", entry)
            entry = re.sub(r"\s+", " ", entry).strip()
            cleaned_entries.append(entry)
        return cleaned_entries
    return []

# Categorize species
def categorize_species(species_list):
    categories = {
        'Large ruminants': ['Cattle', 'Buffalo'],
        'Small ruminants': ['Sheep', 'Goat'],
        'Camelids': ['Dromedary', 'Bactrias', 'Llama', 'Alpaca', 'Vicuna'],
        'Other': ['Reindeer', 'Yak', 'Horse', 'Donkey', 'Duck', 'Bison', 'Pig', 'Dog', 'Chicken']
    }
    species_categories = {category: [] for category in categories}
    for species in species_list:
        for category, species_names in categories.items():
            if species in species_names:
                species_categories[category].append(species)
                break
    return species_categories


# Mapping of combinations to variant names
variant_mapping = {
    'Other': 'Variant1-1',
    'Small ruminants': 'Variant1-2',
    'Large ruminants': 'Variant1-3',
    'Camelids': 'Variant1-4',
    'Other + Small ruminants': 'Variant2-1',
    'Other + Large ruminants': 'Variant2-2',
    'Other + Camelids': 'Variant2-3',
    'Small ruminants + Large ruminants': 'Variant2-4',
    'Small ruminants + Camelids': 'Variant2-5',
    'Large ruminants + Camelids': 'Variant2-6',
    'Other + Small ruminants + Large ruminants': 'Variant3-1',
    'Other + Small ruminants + Camelids': 'Variant3-2',
    'Small ruminants + Large ruminants + Camelids': 'Variant3-3',
    'Other + Small ruminants + Large ruminants + Camelids': 'Variant4'
}

# Function to determine the variant name based on species categories
def determine_variant_name(row):
    categories_present = []
    if row['num_other'] > 0:
        categories_present.append('Other')
    if row['num_small_ruminants'] > 0:
        categories_present.append('Small ruminants')
    if row['num_large_ruminants'] > 0:
        categories_present.append('Large ruminants')
    if row['num_camelids'] > 0:
        categories_present.append('Camelids')
    combination = ' + '.join(categories_present)
    return variant_mapping.get(combination, 'Unknown')

# Process the DataFrame
def process_pastoralist_data(df):
    # Convert all column names to lowercase
    df.columns = df.columns.str.lower()
    
    # Drop the "upload to pastoralist map" column
    df = df.drop(columns=['upload to pastoralist map'])
    
    # Columns to split
    columns_to_split = ['country', 'othernames', 'language', 'species']
    for col in columns_to_split:
        df[col] = df[col].apply(split_to_array)
    
    # Clean location and other columns
    df['location'] = df['location'].apply(clean_location)
    df['breeds'] = df.apply(lambda row: parse_breeds(row['breeds'], row['species']), axis=1)
    df['credit'] = df['credit'].apply(parse_credits)
    
    # Ensure lat and lon columns are numeric
    df[['lat', 'lon']] = df[['lat', 'lon']].apply(pd.to_numeric, errors='coerce')
    
    # Drop rows with missing coordinates
    df = df.dropna(subset=['lat', 'lon'])

    # Calculate the number of species
    df['num_species'] = df['species'].apply(len)
    
    # Categorize species and add a column for the number of categories
    df['species_categories'] = df['species'].apply(categorize_species)
    
    # Add columns for each species category with the number of species in that category
    df['num_large_ruminants'] = df['species_categories'].apply(lambda x: len(x['Large ruminants']))
    df['num_small_ruminants'] = df['species_categories'].apply(lambda x: len(x['Small ruminants']))
    df['num_camelids'] = df['species_categories'].apply(lambda x: len(x['Camelids']))
    df['num_other'] = df['species_categories'].apply(lambda x: len(x['Other']))

    # Add a column for the number of categories
    df['num_categories'] = df.apply(lambda row: sum([1 for category in ['num_large_ruminants', 'num_small_ruminants', 'num_camelids', 'num_other'] if row[category] > 0]), axis=1)

    # Add the VariantName column
    df['variantname'] = df.apply(determine_variant_name, axis=1)

    # Order values within country in alphabetical order
    df['country'] = df['country'].apply(sorted)

    # Convert all column names to lowercase again to ensure consistency
    df.columns = df.columns.str.lower()

    # Create GeoJSON features
    def create_feature(row):
        properties = {col: row[col] for col in df.columns if col not in ['lat', 'lon', 'geometry']}
        geometry = {
            "type": "Point",
            "coordinates": [row['lon'], row['lat']]
        }
        return {"type": "Feature", "geometry": geometry, "properties": properties}

    geojson_features = df.apply(create_feature, axis=1).tolist()

    # Create GeoJSON structure
    return {
        "type": "FeatureCollection",
        "features": geojson_features
    }

In [ ]:
# Read the data
past = pd.read_csv("../data/raw/pastoralists_data/Pastoralists-Data transfer All.csv")

# Process the data
geojson_data = process_pastoralist_data(past)

# Save to a GeoJSON file
output_path = "../data/processed/"
with open(output_path + "pastoralists.geojson", "w") as f:
    json.dump(geojson_data, f, indent=2)

In [ ]:
import geopandas as gpd

a = gpd.read_file("../data/processed/pastoralists.geojson")

# Convert to DataFrame and display
a = pd.DataFrame(a)
a

In [ ]:
# Extract unique species
unique_species = set()

# Iterate over the 'species' column in the DataFrame
for species_list in a["species"].dropna():  # Drop NaNs to avoid errors
    if isinstance(species_list, list):  # Ensure it's a list before processing
        unique_species.update(species_list)

# Convert set to sorted list
unique_species_list = sorted(unique_species)

# Save to a text file
with open("../data/processed/unique_species.txt", "w") as f:
    for species in unique_species_list:
        f.write(species + "\n")